# Save & Load Cells for Your DistilBERT Model
Add these cells at the end of your existing notebook after training is done.

---
## CELL A — Save Everything (Run this once after training)

This saves:
- The model weights (what it learned)
- The tokenizer (so you don't re-download it)
- A config file (architecture details)

All saved in one folder called `hate_speech_model`

In [1]:
import os

SAVE_PATH = './hate_speech_model'  # folder name — you can change this
os.makedirs(SAVE_PATH, exist_ok=True)

# Save model weights + config together
model.save_pretrained(SAVE_PATH)

# Save tokenizer too (so no re-download needed)
tokenizer.save_pretrained(SAVE_PATH)

print(f'Model saved to: {SAVE_PATH}')
print(f'Files saved:')
for f in os.listdir(SAVE_PATH):
    size = os.path.getsize(f'{SAVE_PATH}/{f}') / (1024*1024)
    print(f'  {f}  ({size:.1f} MB)')

# Expected output:
# config.json          (0.0 MB)  — model architecture info
# tokenizer.json       (0.7 MB)  — tokenizer vocabulary
# tokenizer_config.json(0.0 MB)  — tokenizer settings
# vocab.txt            (0.2 MB)  — word vocabulary
# model.safetensors    (255 MB)  — the actual trained weights (biggest file)

NameError: name 'model' is not defined

---
## CELL B — Load Everything (Run this when you come back)

Next time you open Jupyter, just run this cell.
No retraining needed. Loads in ~10-15 seconds.

In [ ]:
# ============================================================
# NEXT SESSION — Run only these imports + this cell
# You do NOT need to re-run training cells
# ============================================================

import torch
import numpy as np
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification

SAVE_PATH = './hate_speech_model'  # same folder name as above

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load tokenizer from saved folder
tokenizer = DistilBertTokenizer.from_pretrained(SAVE_PATH)

# Load model from saved folder
model = DistilBertForSequenceClassification.from_pretrained(SAVE_PATH)
model = model.to(device)
model.eval()  # set to evaluation mode — important!

print('Model loaded successfully!')
print(f'Device: {device}')
# Ready to use — no retraining needed

---
## CELL C — Quick Prediction Test (to verify loading worked)

Run this right after Cell B to confirm the model is working correctly.

In [ ]:
import re
from deep_translator import GoogleTranslator

translator = GoogleTranslator(source='auto', target='en')

def light_clean(text):
    text = str(text)
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    text = re.sub(r'[^a-zA-Z\s.,!?]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()

def predict(text):
    """
    Full pipeline:
    Hindi text → translate → clean → tokenize → predict
    """
    # Step 1: Translate
    english = translator.translate(text)

    # Step 2: Clean
    cleaned = light_clean(english)

    # Step 3: Tokenize
    inputs = tokenizer(
        cleaned,
        return_tensors='pt',
        max_length=128,
        truncation=True,
        padding='max_length'
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Step 4: Predict
    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()[0]
    pred  = int(np.argmax(probs))

    print(f'Input     : {text}')
    print(f'Translated: {english}')
    print(f'Prediction: {"🚨 HATE SPEECH" if pred == 1 else "✅ NOT HATE"}')
    print(f'Confidence: {probs[pred]*100:.1f}%')
    print('-' * 50)

# Test with same sentences from your original notebook
predict('आज मौसम बहुत अच्छा है')                      # Should be NOT HATE
predict('कुत्ते की तरह भौंकना बंद कर')                # Should be HATE
predict('मैं सबसे प्यार करता हूं')                    # Should be NOT HATE

---
## Where is the saved folder on your computer?

The `hate_speech_model` folder is saved in the **same directory** as your Jupyter notebook.

```
Your project folder/
├── Untitled.ipynb              ← your existing notebook
├── translated_hindi_dataset.csv
└── hate_speech_model/          ← NEW folder created by Cell A
    ├── config.json             (architecture)
    ├── tokenizer.json          (vocabulary)
    ├── tokenizer_config.json
    ├── vocab.txt
    └── model.safetensors       (trained weights — ~255MB)
```

## Backup tip:
Copy the `hate_speech_model` folder to Google Drive or a USB drive.
That way even if your laptop crashes, your trained model is safe.

## What to tell your professor:
> 'After fine-tuning, I saved the model using HuggingFace's save_pretrained() method,
> which serializes both the model weights and tokenizer vocabulary to disk.
> This allows the model to be reloaded without retraining using from_pretrained()
> pointing to the local directory instead of the HuggingFace hub.'